# ARC-AGI-3 Solver â€” GPT-OSS-120B â€” 25-Game P1 Public Eval (v16: harmony parser guard)

Public/offline evaluation is overridden to the same 25 public games Ã— 1 pass shape. Competition reruns still use the live private game list from the Kaggle gateway.


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# GPT-OSS-120B / Kaggle input configuration
DATASET_SOURCES: list[str] = [
    "jakobbrggen/taaf-kaggle-source-anim-20260807-anim",
    "driessmit1/arc3-vllm-h100-wheelhouse-v3",
]
# philipvonderlind/vllm-deps provides the tiktoken encoding files
# (cl100k_base.tiktoken + o200k_base.tiktoken) that openai_harmony's Rust core
# loads from TIKTOKEN_ENCODINGS_BASE when offline. The openai-harmony and
# tiktoken *wheels* are already in the H100 wheelhouse (requirements.lock pins
# openai-harmony==0.0.8 and tiktoken==0.12.0), so only the encoding data files
# come from this kernel source.
KERNEL_SOURCES: list[str] = ["philipvonderlind/vllm-deps"]

# Public Kaggle Model: danielhanchen/gpt-oss-120b (Transformers/default/1).
# Kaggle lowercases the framework directory when mounting, so the physical
# path uses "transformers" â€” same as the gregkamradt template's MODEL_ID.
GPTOSS_MODEL_OWNER = "danielhanchen"
GPTOSS_MODEL_SLUG = "gpt-oss-120b"
GPTOSS_MODEL_REF = f"{GPTOSS_MODEL_OWNER}/{GPTOSS_MODEL_SLUG}"
GPTOSS_MODEL_FRAMEWORK = "transformers"
GPTOSS_MODEL_VARIATION = "default"
GPTOSS_MODEL_VERSION = "1"
GPTOSS_SERVED_MODEL_NAME = "gpt-oss-120b"
GPTOSS_MODEL_PATH = Path(
    f"/kaggle/input/models/{GPTOSS_MODEL_OWNER}/{GPTOSS_MODEL_SLUG}/"
    f"{GPTOSS_MODEL_FRAMEWORK}/{GPTOSS_MODEL_VARIATION}/{GPTOSS_MODEL_VERSION}"
)

DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path

    # Prefer the attached bundle whose marker actually exists.
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent

    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(
        json.dumps(data, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Verify the GPT-OSS-120B Kaggle Model before any expensive setup work starts.
if not GPTOSS_MODEL_PATH.is_dir():
    # Fall back to a scan in case Kaggle preserved different path casing.
    _models_root = Path(
        f"/kaggle/input/models/{GPTOSS_MODEL_OWNER}/{GPTOSS_MODEL_SLUG}"
    )
    if _models_root.is_dir():
        for _candidate in _models_root.rglob("config.json"):
            if _candidate.is_file():
                GPTOSS_MODEL_PATH = _candidate.parent
                break
if not GPTOSS_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        "GPT-OSS-120B Kaggle Model is not attached.\n"
        f"Expected path:\n{GPTOSS_MODEL_PATH}\n\n"
        "Attach: danielhanchen/gpt-oss-120b â†’ Transformers â†’ default â†’ Version 1"
    )

_gptoss_config = json.loads(
    (GPTOSS_MODEL_PATH / "config.json").read_text(encoding="utf-8")
)
# vLLM auto-enables the harmony serving path only for model_type == "gpt_oss".
if _gptoss_config.get("model_type") != "gpt_oss":
    raise RuntimeError(
        "Attached model is not gpt-oss "
        f"(config.json model_type={_gptoss_config.get('model_type')!r})."
    )
_gptoss_safetensors = sorted(GPTOSS_MODEL_PATH.glob("*.safetensors"))
if not _gptoss_safetensors:
    raise RuntimeError("GPT-OSS mount has no safetensors weights.")
if not (GPTOSS_MODEL_PATH / "tokenizer.json").is_file() and not (
    GPTOSS_MODEL_PATH / "tokenizer_config.json"
).is_file():
    raise RuntimeError("GPT-OSS mount has no tokenizer files.")

# openai_harmony's Rust core loads the o200k_base/cl100k_base encodings from
# TIKTOKEN_ENCODINGS_BASE when offline (HF_HUB_OFFLINE=1 blocks openaipublic).
TIKTOKEN_ENCODING_FILES = ("cl100k_base.tiktoken", "o200k_base.tiktoken")


def _find_tiktoken_encodings_dir() -> Path:
    """Locate the directory shipping both tiktoken encoding files."""
    for root in (Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()):
        if not root.exists():
            continue
        for candidate in root.rglob("*"):
            if not candidate.is_dir():
                continue
            if all(
                (candidate / name).is_file() for name in TIKTOKEN_ENCODING_FILES
            ):
                return candidate
    raise FileNotFoundError(
        "Could not find a directory containing both "
        + " and ".join(TIKTOKEN_ENCODING_FILES)
        + ". Attach kernel source philipvonderlind/vllm-deps."
    )


TIKTOKEN_ENCODINGS_DIR = _find_tiktoken_encodings_dir()

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# The bundled setup resolver asks for owner/slug. Give it a model ref that maps
# directly to the full Kaggle Model version directory.
kaggle_input_paths[GPTOSS_MODEL_REF] = str(GPTOSS_MODEL_PATH)

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
    "TAAF_GPTOSS_MODEL_REF": GPTOSS_MODEL_REF,
    "TAAF_GPTOSS_MODEL_PATH": str(GPTOSS_MODEL_PATH),
    "TAAF_GPTOSS_SERVED_MODEL_NAME": GPTOSS_SERVED_MODEL_NAME,
    # Must reach the vLLM server process: setup_env -> os.environ ->
    # _command_env() -> heredoc os.environ -> vllm_env() copies os.environ.
    "TIKTOKEN_ENCODINGS_BASE": str(TIKTOKEN_ENCODINGS_DIR),
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)

print("\nâœ… GPT-OSS-120B input configuration ready")
print(f"Model ref:       {GPTOSS_MODEL_REF}")
print(f"Physical path:   {GPTOSS_MODEL_PATH}")
print(f"Served model:    {GPTOSS_SERVED_MODEL_NAME}")
print(f"model_type:      {_gptoss_config.get('model_type')}")
print(f"Safetensors:     {len(_gptoss_safetensors)}")
print(f"Tiktoken dir:    {TIKTOKEN_ENCODINGS_DIR}")
print(f"TAAF input map:  {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


In [ ]:
# Audit the attached inputs that matter for this run.
print("=== TAAF bundle ===")
print(BUNDLE_DIR)
print("Exists:", BUNDLE_DIR.exists())

print("\n=== vLLM wheelhouse ===")
_vllm_wheelhouse = Path(
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3"
)
print(_vllm_wheelhouse)
print("Exists:", _vllm_wheelhouse.exists())

print("\n=== GPT-OSS-120B Kaggle Model ===")
print(GPTOSS_MODEL_PATH)
print("Exists:", GPTOSS_MODEL_PATH.exists())
print("Safetensors:", len(list(GPTOSS_MODEL_PATH.glob("*.safetensors"))))
print("model_type:", _gptoss_config.get("model_type"))

print("\n=== tiktoken encodings (openai_harmony offline) ===")
print(TIKTOKEN_ENCODINGS_DIR)
print(
    "Both files present:",
    all(
        (TIKTOKEN_ENCODINGS_DIR / name).is_file()
        for name in TIKTOKEN_ENCODING_FILES
    ),
)


In [ ]:
import re


def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []

    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env["HF_HUB_OFFLINE"] = "1"
    env["TRANSFORMERS_OFFLINE"] = "1"
    env.update(_load_setup_env())
    return env


def _replace_python_assignment(
    command: str,
    variable_name: str,
    value: str,
) -> tuple[str, int]:
    """Replace a top-level Python string assignment inside the setup here-doc."""
    pattern = rf"(?m)^{re.escape(variable_name)}\s*=\s*(['\"])[^\r\n]*?\1\s*$"
    replacement = f"{variable_name} = {value!r}"
    return re.subn(pattern, replacement, command, count=1)


# V16 (2026-09-01): root cause CONFIRMED. Not async-scheduling, not fp8 KV,
# not the vllm version â€” gpt-oss occasionally emits a malformed harmony
# continuation, and vLLM 0.19.0's parse_output_into_messages has no error
# handling, so HarmonyError 500s the whole request. (v14 canary: 7/37 failed
# under the proven gregkamradt flags, including 0/1 at concurrency 1; a forum
# post on this exact wheelhouse+model measured 8.1% and shared this fix.)
# v16 patches the installed harmony_utils.py before the server starts (see
# _V16_GUARD_DEF below); the canary now distinguishes hard (HTTP 500, fatal)
# from soft (truncated turn, expected ~8%) failures.
#
# V14 serving flags (2026-09-01, harmony 500 fix). v13 ran --async-scheduling +
# CUDA graphs + the default --max-num-seqs (256) and failed massively under
# load: the harmony StreamableParser rejected the model's own generated tokens
# ("Unexpected token N while expecting start token 200006") on most analyzer
# requests at ~25-way concurrency, while the single-request temperature-0
# smoke test passed and the engine stayed healthy. The varying leading token on
# each failure is the signature of dropped leading generated tokens â€” an
# async-scheduling/detokenizer race that Qwen's tolerant text-based tool parser
# absorbed in V4 but harmony's strict Rust parser cannot. V14 therefore matches
# the proven gregkamradt/arc-agi-3-gpt-oss-120b serving shape exactly:
#   * --enforce-eager + --max-num-seqs 12 (no async scheduling, capped batches)
#   * --tool-call-parser 'openai' and --kv-cache-dtype fp8 kept from v13
#   * --generation-config / --enable-prefix-caching dropped (prefix caching is
#     on by default in vLLM 0.19 either way; sampling params come per request)
# The cost is decode throughput (eager). Re-adding performance flags happens
# only after a clean run, one flag at a time, with the harmony canary guarding.
#
# The bundled start_vllm_server() builds its arg list as a Python list literal
# inside setup_commands.json, so patch it as a string the same way the model
# identity is patched. That keeps one source of truth for the serving flags.
_V16_SERVING_ANCHOR = """        '--enable-auto-tool-choice',
        '--tool-call-parser',
        'qwen3_coder',
        '--generation-config',
        'vllm',
        '--enable-prefix-caching',
        '--default-chat-template-kwargs',
        '{"preserve_thinking": true}',
        '--reasoning-parser',
        'qwen3',
        '--max-model-len',
        str(VLLM_MAX_MODEL_LEN),
    ]"""

_V16_SERVING_BLOCK = """        '--enable-auto-tool-choice',
        '--tool-call-parser',
        'openai',
        '--max-num-seqs',
        '12',
        '--max-model-len',
        str(VLLM_MAX_MODEL_LEN),
        '--kv-cache-dtype',
        'fp8',
        '--enforce-eager',
    ]"""

# Canary injection anchors (see _inject_harmony_canary below). Both are
# unique substrings of the TAAF setup heredoc in setup_commands.json.
_V16_CANARY_DEF_ANCHOR = "\n\n\nprint(f'vLLM wheelhouse path: {WHEELHOUSE}', flush=True)"

_V16_CANARY_DEF = r'''def run_vllm_harmony_canary() -> None:
    # Concurrent tool-bearing probe of the harmony (gpt-oss) serving path.
    # v13 failed exactly here: the single-request temperature-0 smoke test
    # above passed, then most analyzer requests at ~25-way concurrency and
    # temperature 1.0 died with harmony parse errors ("Unexpected token N
    # while expecting start token 200006") while the engine stayed healthy.
    # v14 then proved the failure is per-request probabilistic at temperature
    # 1.0 (7/37 failed, including 0/1 at concurrency 1, under the proven
    # gregkamradt flags): gpt-oss occasionally emits a malformed harmony
    # continuation, and vLLM 0.19.0's parse_output_into_messages turns that
    # into an HTTP 500 (whole turn lost). v16 patches that parser (see
    # patch_harmony_parser_guard), so a healthy server now answers every
    # request with HTTP 200; a malformed turn merely comes back truncated
    # (soft: no tool call). This canary therefore splits HARD failures
    # (HTTP/transport errors, still fatal) from SOFT ones (truncated turns,
    # expected at a small rate and non-fatal), so it aborts only when the
    # server is genuinely broken.
    import concurrent.futures

    tools = [{
        'type': 'function',
        'function': {
            'name': 'submit_answer',
            'description': 'Submit the computed result.',
            'parameters': {
                'type': 'object',
                'properties': {'answer': {'type': 'string'}},
                'required': ['answer'],
            },
        },
    }]

    def one_request(i: int) -> str | None:
        payload = {
            'model': SERVED_MODEL_NAME,
            'messages': [
                {'role': 'system', 'content': 'You are a precise assistant.'},
                {
                    'role': 'user',
                    'content': (
                        'Compute ' + str(17 * (i + 3)) + ' and submit the result '
                        'with the submit_answer tool. Keep the reasoning short.'
                    ),
                },
            ],
            'temperature': 1.0,
            'top_p': 1.0,
            'max_tokens': 512,
            'tools': tools,
            'tool_choice': 'auto',
            'chat_template_kwargs': {'enable_thinking': False},
        }
        response = request_json(
            VLLM_BASE_URL + '/chat/completions', payload=payload, timeout=900
        )
        choice = response['choices'][0]
        tool_calls = choice['message'].get('tool_calls') or []
        if not tool_calls:
            return 'no tool call (finish_reason=' + repr(choice.get('finish_reason')) + ')'
        return None

    hard_failures = 0
    soft_failures = 0
    for level in (1, 12, 24):
        with concurrent.futures.ThreadPoolExecutor(max_workers=level) as pool:
            futures = [pool.submit(one_request, i) for i in range(level)]
            level_hard = []
            level_soft = []
            for i, future in enumerate(futures):
                try:
                    problem = future.result()
                    hard = False
                except Exception as exc:
                    problem = type(exc).__name__ + ': ' + str(exc)[:300]
                    hard = True
                if problem is not None:
                    if hard:
                        level_hard.append((i, problem))
                    else:
                        level_soft.append((i, problem))
        ok = level - len(level_hard) - len(level_soft)
        print('harmony canary: concurrency ' + str(level) + ': ' + str(ok) + '/' + str(level) + ' ok (' + str(len(level_hard)) + ' hard, ' + str(len(level_soft)) + ' soft)', flush=True)
        for i, problem in level_hard:
            print('  canary HARD failure [level ' + str(level) + ', request ' + str(i) + ']: ' + problem, flush=True)
        for i, problem in level_soft:
            print('  canary soft failure [level ' + str(level) + ', request ' + str(i) + ']: ' + problem, flush=True)
        hard_failures += len(level_hard)
        soft_failures += len(level_soft)

    if hard_failures == 0:
        print('harmony canary PASSED: 0 hard failures of 37; ' + str(soft_failures) + ' soft (truncated) turn(s)', flush=True)
    elif hard_failures >= 3:
        raise RuntimeError(
            'harmony canary: ' + str(hard_failures) + ' of 37 requests hard-failed '
            '(HTTP 500 / transport) even with the parser guard applied; the vLLM '
            'server is broken. Aborting before the benchmark burns the GPU budget.'
        )
    else:
        print('WARNING: harmony canary saw ' + str(hard_failures) + ' hard failure(s); proceeding.', flush=True)
    if soft_failures >= 12:
        print('WARNING: ' + str(soft_failures) + '/37 canary turns came back truncated (no tool call); the parser guard is degrading many turns â€” watch per-game behavior.', flush=True)'''

_V16_CANARY_DEF_BLOCK = "\n\n\n" + _V16_CANARY_DEF + _V16_CANARY_DEF_ANCHOR

_V16_CANARY_CALL_ANCHOR = "start_vllm_server()\nrun_vllm_api_smoke_test()\n"

_V16_CANARY_CALL_BLOCK = "start_vllm_server()\nrun_vllm_api_smoke_test()\nrun_vllm_harmony_canary()\n"

# v16 harmony parser guard: patch text + heredoc insertion anchors
# (see _inject_harmony_guard below). The def anchor is the same wheelhouse
# print the canary uses; both injections preserve that print, so the order
# of the two injections does not matter.
_V16_GUARD_DEF = r'''
def patch_harmony_parser_guard() -> None:
    # gpt-oss occasionally emits a malformed harmony continuation (an <|end|>
    # not followed by <|start|>, or a hallucinated functions.* tool-result
    # turn). vLLM 0.19.0's parse_output_into_messages has no error handling
    # around the StreamableParser loop, so HarmonyError escapes the request
    # handler and the client gets an HTTP 500 and loses the whole turn
    # (measured 8.1% of requests in a public 25-game run on this exact
    # wheelhouse+model; our v14 canary saw 7/37 including a single
    # non-concurrent request). Both non-streaming consumers (parse_chat_output
    # and the openai tool parser's extract_tool_calls) route through this one
    # function, so one guard covers both. With it, malformed turns degrade to
    # truncated-but-usable responses instead of 500s.
    path = SITE_PACKAGES / 'vllm' / 'entrypoints' / 'openai' / 'parser' / 'harmony_utils.py'
    if not path.is_file():
        raise FileNotFoundError('harmony_utils.py not found at ' + str(path))
    old = (
        'def parse_output_into_messages(token_ids: Iterable[int]) -> StreamableParser:\n'
        '    parser = get_streamable_parser_for_assistant()\n'
        '    for token_id in token_ids:\n'
        '        parser.process(token_id)\n'
        '    return parser'
    )
    new = (
        'def parse_output_into_messages(token_ids: Iterable[int]) -> StreamableParser:\n'
        '    parser = get_streamable_parser_for_assistant()\n'
        '    for token_id in token_ids:\n'
        '        try:\n'
        '            parser.process(token_id)\n'
        '        except Exception:\n'
        '            # malformed harmony continuation: truncate this turn instead\n'
        '            # of raising HarmonyError out of the request handler (which\n'
        '            # 500s the whole request and loses the turn).\n'
        '            break\n'
        '    return parser'
    )
    text = path.read_text(encoding='utf-8')
    if text.count(old) == 0 and text.count(new) == 1:
        print('harmony parser guard: already applied', flush=True)
        return
    count = text.count(old)
    if count != 1:
        raise RuntimeError(
            'harmony parser guard: expected exactly 1 patch target in '
            'harmony_utils.py, found ' + str(count) + ' (vLLM layout changed?)'
        )
    path.write_text(text.replace(old, new, 1), encoding='utf-8')
    compile(path.read_text(encoding='utf-8'), str(path), 'exec')
    print('harmony parser guard: applied (malformed harmony turns now truncate '
          'instead of HTTP 500)', flush=True)
'''

_V16_GUARD_DEF_ANCHOR = "\n\n\nprint(f'vLLM wheelhouse path: {WHEELHOUSE}', flush=True)"

_V16_GUARD_DEF_BLOCK = "\n\n\n" + _V16_GUARD_DEF + _V16_GUARD_DEF_ANCHOR

_V16_GUARD_CALL_ANCHOR = "def start_vllm_server() -> None:\n    install_vllm_wheelhouse()\n"

_V16_GUARD_CALL_BLOCK = (
    "def start_vllm_server() -> None:\n"
    "    install_vllm_wheelhouse()\n"
    "    patch_harmony_parser_guard()\n"
)



def _inject_harmony_canary(commands: list[str]) -> list[str]:
    # Insert run_vllm_harmony_canary() into the TAAF setup heredoc so it runs
    # right after the bundled smoke test, while the server is otherwise idle.
    # See the V14 comment above for why the bundled smoke test is not enough.
    patched = []
    for command in commands:
        if "def run_vllm_api_smoke_test()" in command:
            already = (
                command.count(_V16_CANARY_DEF) == 1
                and command.count(_V16_CANARY_CALL_BLOCK) == 1
            )
            if not already:
                if command.count(_V16_CANARY_DEF) or command.count(
                    _V16_CANARY_CALL_BLOCK
                ):
                    raise RuntimeError("harmony canary is only partially applied")
                if command.count(_V16_CANARY_DEF_ANCHOR) != 1:
                    raise RuntimeError(
                        "harmony canary definition insertion point not found"
                    )
                if command.count(_V16_CANARY_CALL_ANCHOR) != 1:
                    raise RuntimeError("harmony canary call insertion point not found")
                command = command.replace(
                    _V16_CANARY_DEF_ANCHOR, _V16_CANARY_DEF_BLOCK, 1
                )
                command = command.replace(
                    _V16_CANARY_CALL_ANCHOR, _V16_CANARY_CALL_BLOCK, 1
                )
        patched.append(command)
    return patched


def _inject_harmony_guard(commands: list[str]) -> list[str]:
    # Insert patch_harmony_parser_guard() into the TAAF setup heredoc: the def
    # lands next to the canary def, and the call lands inside
    # start_vllm_server() immediately after install_vllm_wheelhouse(), so the
    # installed file is patched before the server process ever imports it.
    patched = []
    for command in commands:
        if "def run_vllm_api_smoke_test()" in command:
            already = (
                command.count(_V16_GUARD_DEF) == 1
                and command.count(_V16_GUARD_CALL_BLOCK) == 1
            )
            if not already:
                if command.count(_V16_GUARD_DEF) or command.count(_V16_GUARD_CALL_BLOCK):
                    raise RuntimeError("harmony parser guard is only partially applied")
                if command.count(_V16_GUARD_DEF_ANCHOR) != 1:
                    raise RuntimeError(
                        "harmony guard definition insertion point not found"
                    )
                if command.count(_V16_GUARD_CALL_ANCHOR) != 1:
                    raise RuntimeError("harmony guard call insertion point not found")
                command = command.replace(
                    _V16_GUARD_DEF_ANCHOR, _V16_GUARD_DEF_BLOCK, 1
                )
                command = command.replace(
                    _V16_GUARD_CALL_ANCHOR, _V16_GUARD_CALL_BLOCK, 1
                )
        patched.append(command)
    return patched


def _patch_gptoss_setup_commands(commands: list[str]) -> list[str]:
    """
    Preserve the TAAF deployment setup but replace its model identity with the
    GPT-OSS-120B Kaggle Model. This avoids copying/forking the large bundled
    setup script and keeps the wheelhouse/GPU/vLLM behavior from the source
    bundle.
    """
    patched: list[str] = []
    replacement_counts = {
        "MODEL_OWNER": 0,
        "MODEL_SLUG": 0,
        "SERVED_MODEL_NAME": 0,
        "SERVING_FLAGS": 0,
    }

    replacements = {
        "MODEL_OWNER": GPTOSS_MODEL_OWNER,
        "MODEL_SLUG": GPTOSS_MODEL_SLUG,
        "SERVED_MODEL_NAME": GPTOSS_SERVED_MODEL_NAME,
    }

    for raw_command in commands:
        command = str(raw_command)

        for variable_name, value in replacements.items():
            command, count = _replace_python_assignment(
                command,
                variable_name,
                value,
            )
            replacement_counts[variable_name] += count

        # Make offline behavior explicit in the child process as well.
        if "def vllm_env()" in command:
            command = command.replace(
                "'VLLM_NO_USAGE_STATS': '1',",
                "'VLLM_NO_USAGE_STATS': '1',\n"
                "            'HF_HUB_OFFLINE': '1',\n"
                "            'TRANSFORMERS_OFFLINE': '1',",
                1,
            )

        # Rework the vLLM launch args for gpt-oss (see _V16_SERVING_BLOCK above).
        if "def start_vllm_server()" in command:
            found = command.count(_V16_SERVING_ANCHOR)
            if found != 1:
                raise RuntimeError(
                    "Expected exactly one vLLM arg list to extend in the bundled "
                    f"setup command; found {found}. The attached TAAF bundle's "
                    "start_vllm_server() has changed -- re-derive the anchor from "
                    "the 'taaf.kaggle: setup command:' echo in a previous run log."
                )
            command = command.replace(_V16_SERVING_BLOCK, "", 1)  # idempotent
            command = command.replace(_V16_SERVING_ANCHOR, _V16_SERVING_BLOCK, 1)
            replacement_counts["SERVING_FLAGS"] += 1

        patched.append(command)

    missing = [
        name for name, count in replacement_counts.items() if count == 0
    ]
    if missing:
        raise RuntimeError(
            "Could not update the bundled TAAF setup for GPT-OSS-120B. "
            "Missing assignment(s): "
            + ", ".join(missing)
            + ". The attached TAAF bundle's setup_commands.json has changed."
        )

    print("taaf.kaggle: GPT-OSS setup patch =", replacement_counts, flush=True)
    return patched


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return

    commands = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(commands, list):
        raise RuntimeError(f"{path} must contain a JSON list of shell commands.")

    if filename == "setup_commands.json":
        commands = _patch_gptoss_setup_commands(commands)
        commands = _inject_harmony_canary(commands)
        commands = _inject_harmony_guard(commands)

    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(
            str(command),
            shell=True,
            check=check,
            cwd=WORKING_DIR,
            env=env,
        )
        if not check and result.returncode != 0:
            print(
                f"taaf.kaggle: {label} command exited with {result.returncode}",
                flush=True,
            )

        # Setup commands may export additional runtime settings.
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text(
        "".join(f"{entry}\n" for entry in source_entries),
        encoding="utf-8",
    )
    print(
        f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)",
        flush=True,
    )

# Run the TAAF deployment setup, patched to use GPT-OSS-120B.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# gpt-oss sampling overrides (OpenAI's recommended operating point for the
# family: temperature 1.0, top_p 1.0). Applied AFTER the setup commands because
# the bundle's heredoc re-writes these keys in TAAF_KAGGLE_SETUP_ENV (its
# existing.update wins over any value written earlier). The analyzer reads them
# as module-level constants in inference/agent/tool_agent.py at first import â€”
# which happens when benchmark_initial.pkl is unpickled in a later cell â€” so
# os.environ set here is what the analyzer picks up. top_k=0 makes
# build_chat_payload omit top_k from the request payload entirely.
_SAMPLING_OVERRIDES = {
    "LOCAL_ANALYZER_TEMPERATURE": "1.0",
    "LOCAL_ANALYZER_TOP_P": "1.0",
    "LOCAL_ANALYZER_TOP_K": "0",
}
os.environ.update(_SAMPLING_OVERRIDES)
_write_setup_env_updates(_SAMPLING_OVERRIDES)
print("taaf.kaggle: gpt-oss sampling overrides =", _SAMPLING_OVERRIDES)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [
    entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry
]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

# Fail early if the analyzer is still exposing an old model identity.
_actual_model_id = os.environ.get("INFERENCE_ANALYZER_MODEL", "")
if _actual_model_id != GPTOSS_SERVED_MODEL_NAME:
    raise RuntimeError(
        "TAAF setup completed, but the analyzer model ID is wrong: "
        f"{_actual_model_id!r}; expected {GPTOSS_SERVED_MODEL_NAME!r}"
    )

print("\nâœ… TAAF/vLLM setup completed for GPT-OSS-120B")
print("Model path:", GPTOSS_MODEL_PATH)
print("Analyzer model:", _actual_model_id)
print("Analyzer endpoint:", os.environ.get("LOCAL_ANALYZER_BASE_URL"))


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Inline customization hook â€” Q38 P1-style public evaluation.
#
# Q38 P1 runs the full 25 public ARC-AGI-3 games once each (25 games Ã— 1 pass).
# This override applies only to the public/offline notebook run. Competition reruns
# still replace bm.games from Kaggle's live gateway in the final run cell.

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))
print("GPT-OSS-120B model path:", os.environ.get("TAAF_GPTOSS_MODEL_PATH"))

Q38_P1_PUBLIC_GAME_IDS = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47",
]

if not true_submission:
    if len(Q38_P1_PUBLIC_GAME_IDS) != 25 or len(set(Q38_P1_PUBLIC_GAME_IDS)) != 25:
        raise RuntimeError("Q38 P1 public game list must contain exactly 25 unique games.")
    if not bm.games:
        raise RuntimeError("benchmark_initial.pkl contains no template public game.")

    import taaf.game_api

    template_game = bm.games[0]
    arcade_spec = getattr(template_game, "arcade_spec", None)
    if arcade_spec is None:
        arcade_spec = getattr(template_game, "_arcade_spec", None)
    if arcade_spec is None:
        raise RuntimeError(
            "Could not recover the public ArcadeSpec from benchmark_initial.pkl; "
            "cannot construct the 25-game Q38 P1 evaluation set."
        )

    bm.games = [
        taaf.game_api.GameAPI(env_name=game_id, arcade_spec=arcade_spec)
        for game_id in Q38_P1_PUBLIC_GAME_IDS
    ]
    bm.n_passes = 1
    bm.game_weights = None

    if hasattr(bm.solver, "concurrency"):
        bm.solver.concurrency = 28

    # THROUGHPUT DIAGNOSTIC (2026-08-31) â€” was a hardcoded 7920.0.
    #
    # 7920 s is correct for the ~110-game submission: 110 / 28 concurrency = 4 waves
    # x 2.2 h = 8.8 h, which fills the GPU budget exactly. It is simply wrong for a
    # 25-game public run, which is 25 / 28 = ONE wave: the previous run left 74.7%
    # of the budget idle and every one of the 25 games ended `state=gave_up` on the
    # clock at ~52 actions, none from solver failure or game-over.
    #
    # So derive the cap from the real soft deadline instead of hardcoding a second
    # magic number. Only ever raises (max against 7920), only ever runs on the
    # public path (this whole block is inside `if not true_submission`), and
    # bm.run(soft_end_time=soft_end) remains the hard backstop.
    if hasattr(bm.solver, "max_runtime_s_per_game"):
        _concurrency = max(1, int(getattr(bm.solver, "concurrency", 1) or 1))
        _waves = -(-len(bm.games) // _concurrency)
        _cap = 7920.0
        if soft_end is not None:
            # Reserve time after bm.run() for diagnostics/movies/pickles. Measured
            # ~300 s for 1298 actions; 2400 s covers a ~4x larger run with margin.
            _artifact_margin_s = 2400.0
            _window_s = (soft_end - datetime.now()).total_seconds() - _artifact_margin_s
            if _window_s > 0:
                _cap = max(_cap, _window_s / _waves)
        bm.solver.max_runtime_s_per_game = _cap
        print(
            f"Runtime cap derived: {len(bm.games)} games / concurrency {_concurrency} "
            f"= {_waves} wave(s) -> {_cap:.0f}s per game "
            f"({_cap / 7920.0:.2f}x the previous 7920s), soft_end={soft_end}"
        )

    bm.label = f"{bm.label}-25g-p1"
    print(f"Public evaluation override: {len(bm.games)} games Ã— {bm.n_passes} pass = {len(bm.games) * bm.n_passes} runs")
    print("Public evaluation concurrency:", getattr(bm.solver, "concurrency", None))
    print("Public per-game runtime cap (s):", getattr(bm.solver, "max_runtime_s_per_game", None))


# ---------------------------------------------------------------------------
# vLLM health watchdog (V4, 2026-08-31).
#
# A mid-run vLLM death otherwise takes the whole notebook down and zeroes the
# score. This deliberately does NOT relaunch the server: the launch lives in the
# bundle's setup_commands.json subprocess, and duplicating its arg list here would
# create a second source of truth for the serving flags. Instead it converts a hard
# crash into a graceful stop -- setting solver._stop_event makes
# _HarnessGameSession.should_stop() return True (framework/solver.py:246-250), and
# _finish_if_needed() then marks each still-playing game "cancelled" and calls
# finish_game() (solver.py:340-343), so levels already completed still score and
# the artifacts/submission still get written.
#
# Armed on BOTH the public and submission paths on purpose: the submission is the
# run where losing 9 h of GPU to a dead server actually costs something.
_WATCHDOG_POLL_S = 15.0
_WATCHDOG_FAILS_TO_STOP = 4  # ~60 s of sustained failure before giving up


def _vllm_health_watchdog(base_url: str, stop_event) -> None:
    url = base_url.rstrip("/") + "/models"
    consecutive = 0
    announced = False
    while True:
        time.sleep(_WATCHDOG_POLL_S)
        try:
            with urlopen(url, timeout=10) as response:
                healthy = getattr(response, "status", 200) == 200
        except Exception:
            healthy = False

        if healthy:
            if consecutive:
                print(
                    f"watchdog: vLLM recovered after {consecutive} failed check(s)",
                    flush=True,
                )
            consecutive = 0
            announced = False
            continue

        consecutive += 1
        print(
            f"watchdog: vLLM health check {consecutive}/{_WATCHDOG_FAILS_TO_STOP} "
            f"failed ({url})",
            flush=True,
        )
        # Re-set rather than return: _run_games() clears the event when it starts,
        # so a server that died before the run began must be caught again.
        if consecutive >= _WATCHDOG_FAILS_TO_STOP and not stop_event.is_set():
            if not announced:
                print(
                    "watchdog: vLLM unreachable; setting the solver stop event so "
                    "completed levels are still scored.",
                    flush=True,
                )
                announced = True
            stop_event.set()


_watchdog_base_url = os.environ.get("LOCAL_ANALYZER_BASE_URL", "")
_watchdog_event = getattr(bm.solver, "_stop_event", None)
if _watchdog_base_url and _watchdog_event is not None:
    import threading

    threading.Thread(
        target=_vllm_health_watchdog,
        args=(_watchdog_base_url, _watchdog_event),
        name="vllm-health-watchdog",
        daemon=True,
    ).start()
    print(
        f"vLLM watchdog armed: {_watchdog_base_url.rstrip('/')}/models every "
        f"{_WATCHDOG_POLL_S:.0f}s; stop after {_WATCHDOG_FAILS_TO_STOP} "
        "consecutive failures"
    )
else:
    print(
        "vLLM watchdog NOT armed -- base_url="
        f"{_watchdog_base_url!r}, stop_event="
        f"{'present' if _watchdog_event is not None else 'MISSING'}"
    )


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html â€” minimal diagnostics (real submission) suppresses it.")

In [ ]:
# === F/m measurement (injected by _analysis/inject_cell.py) ===
# Reads the benchmark.json this run just wrote and prints the exact RHAE
# decomposition: weighted depth F, action multiple m, waste, lever pricing.
# Fail-soft on purpose: a bug here must not red-flag a 2.5h GPU run.
import json, sys, traceback
from pathlib import Path

import json
import sys
from pathlib import Path



def level_score(baseline, actions):
    if actions > 0 and baseline > 0:
        return min(115.0, (baseline / actions) ** 2 * 100.0)
    return 0.0


def analyse_run(run):
    """Return a dict of measurements for one game run, or None if unscoreable."""
    b = run.get("base_actions_per_level")
    n = run.get("number_of_levels") or 0
    if not b or n == 0:
        return None
    a = list(run.get("actions_per_level") or [])
    a += [0] * (n - len(a))
    b = list(b) + [0] * (n - len(b))
    lc = min(run.get("levels_completed", 0), n)

    total_w = n * (n + 1) // 2
    total_s, max_w, cleared = 0.0, 0, []
    for idx in range(n):
        w = idx + 1
        s = level_score(b[idx], a[idx]) if idx < lc else 0.0
        if s > 0:
            max_w += w
            cleared.append(idx)
        total_s += s * w
    score = min(total_s / total_w, max_w / total_w * 100.0)

    cleared_b = sum(b[i] for i in cleared)
    cleared_a = sum(a[i] for i in cleared)
    return {
        "game_id": run.get("game_id", "?"),
        "n": n,
        "levels_completed": lc,
        "state": run.get("state"),
        "score": score,
        "reported": run.get("final_score"),
        "f": max_w / total_w,           # weighted-depth fraction
        "total_w": total_w,
        "max_w": max_w,
        "b": b,
        "a": a,
        "cleared": cleared,
        "cleared_b": cleared_b,
        "cleared_a": cleared_a,
        "m": (cleared_a / cleared_b) if cleared_b else None,
        "actions_total": sum(a),
        "actions_wasted": sum(a[idx] for idx in range(n) if idx >= lc),
    }


def counterfactual(runs, m_target=None, extra_levels=0, m_for_extra=None):
    """Aggregate score if cleared levels ran at m_target, and/or if each game
    cleared `extra_levels` more levels (those at m_for_extra)."""
    tot = 0.0
    for r in runs:
        n, lc, b = r["n"], r["levels_completed"], r["b"]
        new_lc = min(n, lc + extra_levels)
        total_w = r["total_w"]
        total_s, max_w = 0.0, 0
        for idx in range(n):
            w = idx + 1
            if idx >= new_lc:
                continue
            if idx < lc:
                if m_target:
                    m = m_target
                else:
                    act, base = r["a"][idx], b[idx]
                    m = (act / base) if (act > 0 and base > 0) else 1.0
            else:
                m = m_for_extra if m_for_extra else (r["m"] or 1.0)
            s = level_score(b[idx], max(1.0, m * b[idx]))
            if s > 0:
                max_w += w
            total_s += s * w
        tot += min(total_s / total_w, max_w / total_w * 100.0)
    return tot / len(runs)


def report(path):
    path = Path(path)
    if not path.is_file():
        print("no benchmark.json at %s" % path)
        print("Download it from the public run's /kaggle/working output.")
        return
    data = json.loads(path.read_text(encoding="utf-8"))
    raw = data.get("game_runs") or []
    runs = [x for x in (analyse_run(r) for r in raw) if x]

    print(f"label={data.get('label')}  n_passes={data.get('n_passes')}  "
          f"game_runs={len(raw)}  scoreable={len(runs)}")
    if len(runs) < len(raw):
        print(f"  !! {len(raw)-len(runs)} runs had no baselines (hidden by the engine) "
              "â€” they cannot be scored locally.")
    if not runs:
        return

    # ---- cross-check my formula against the harness's own number -----------
    bad = [r for r in runs if r["reported"] is not None
           and abs(r["score"] - r["reported"]) > 1e-6]
    if bad:
        print(f"\n!! FORMULA MISMATCH on {len(bad)} runs â€” trust `reported`, not this script:")
        for r in bad[:5]:
            print(f"   {r['game_id']}: mine={r['score']:.4f} reported={r['reported']:.4f}")
    else:
        print("formula check: reproduces every reported final_score exactly.")

    # ---- per game ----------------------------------------------------------
    print(f"\n{'game':<18}{'lv':>6}{'score':>8}{'f':>8}{'m':>7}"
          f"{'acts':>7}{'wasted':>8}  per-level m")
    for r in sorted(runs, key=lambda x: -x["score"]):
        upto = min(r["n"], r["levels_completed"] + 1)
        parts = []
        for i in range(upto):
            parts.append("%.1f" % (r["a"][i] / r["b"][i]) if r["b"][i] else "-")
        ms = " ".join(parts)
        m_txt = "%.2f" % r["m"] if r["m"] else "-"
        print("%-18s%3d/%-2d%8.2f%8.1f%%%7s%7d%8d  %s" % (
            r["game_id"][:17], r["levels_completed"], r["n"], r["score"],
            r["f"] * 100, m_txt, r["actions_total"], r["actions_wasted"], ms))

    # ---- aggregate ---------------------------------------------------------
    n = len(runs)
    agg = sum(r["score"] for r in runs) / n
    F = sum(r["f"] for r in runs) / n
    cb = sum(r["cleared_b"] for r in runs)
    ca = sum(r["cleared_a"] for r in runs)
    at = sum(r["actions_total"] for r in runs)
    aw = sum(r["actions_wasted"] for r in runs)
    lv = sum(r["levels_completed"] for r in runs)

    print(f"\n{'='*74}\nAGGREGATE over {n} runs")
    print(f"  score                 {agg:8.3f}   <- compare to the LB number")
    print(f"  F (weighted depth)    {F:8.2%}")
    print(f"  levels cleared        {lv:8d}   ({lv/n:.2f} per game)")
    print(f"  actions spent         {at:8d}   ({at/n:.0f} per game)")
    if at > 0:
        print(f"  actions wasted        {aw:8d}   {aw/at:.1%} went into levels never completed")
    zero = sum(1 for r in runs if r["levels_completed"] == 0)
    print(f"  games at zero         {zero:8d}   {zero/n:.1%}")
    if cb == 0:
        print("\n  Nothing cleared anywhere â€” no m to measure, no levers to price.")
        print("  The bottleneck is depth, not efficiency. Stop here.")
        return
    m_bar = ca / cb
    ident = 100 * F / m_bar ** 2
    print(f"  m (action multiple)   {m_bar:8.2f}   on cleared levels only")
    print(f"  100*F/m^2             {ident:8.3f}   exact only if m were uniform; "
          f"gap to {agg:.2f} = spread in m")

    # ---- price each lever at the depth actually achieved -------------------
    print(f"\n{'='*74}\nLEVER PRICING (measured depth held fixed)")
    print(f"  {'every cleared level forced to m =':<38}{'score':>9}{'vs now':>9}")
    for mt in (m_bar, 2.0, 1.5, 1.2, 1.0):
        v = counterfactual(runs, m_target=mt)
        tag = "  (= aggregate mean m)" if abs(mt - m_bar) < 1e-9 else ""
        print(f"  m = {mt:<34.2f}{v:>9.2f}{v/agg:>8.2f}x{tag}")

    print(f"\n  {'+depth, same m (FREE ACTIONS ASSUMED)':<38}{'score':>9}{'vs now':>9}")
    print("  (upper bound: extra levels cost extra actions we may not have time for)")
    for extra in (1, 2, 3):
        v = counterfactual(runs, extra_levels=extra)
        extra_cost = sum(
            sum(r["b"][i] for i in range(r["levels_completed"],
                                         min(r["n"], r["levels_completed"] + extra)))
            for r in runs) * m_bar / n
        print(f"  +{extra} level(s) every game{'':<18}{v:>9.2f}{v/agg:>8.2f}x"
              f"   (+{extra_cost:.0f} actions/game at m={m_bar:.2f})")

    print(f"\n  {'both levers together':<28}{'score':>9}{'vs now':>9}")
    for extra, mt in ((1, 1.5), (1, 1.2), (2, 1.5), (2, 1.2)):
        v = counterfactual(runs, m_target=mt, extra_levels=extra, m_for_extra=mt)
        print(f"  +{extra} level, m={mt:<16.1f}{v:>9.2f}{v/agg:>8.2f}x")

    print(f"\n{'='*74}\nTARGETS  (score = 100*F/m^2, so these are the isoquants)")
    for target, label in ((2.31, "our best LB draw"), (3.37, "~rank 20"),
                          (4.67, "~rank 5"), (5.99, "rank 1")):
        need_m = (100 * F / target) ** 0.5
        need_F = target * m_bar ** 2 / 100
        print(f"  {target:4.2f} ({label:<17}): at measured F={F:.2%} need m<={need_m:4.2f}"
              f"   |  at measured m={m_bar:.2f} need F>={need_F:5.2%}")

def _find_bench():
    direct = WORKING_DIR / "benchmark.json"   # bm.job_dir = WORKING_DIR
    if direct.is_file():
        return direct
    hits = sorted(WORKING_DIR.rglob("benchmark.json"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    if hits:
        print("benchmark.json not at %s; using %s" % (direct, hits[0]))
        return hits[0]
    return direct

try:
    report(_find_bench())
except Exception:
    print("MEASUREMENT CELL FAILED (run itself is unaffected):")
    traceback.print_exc()
